# Solomon DQN Atari Boxing Notebook

This notebook documents my work on training and evaluating a Deep Q-Network (DQN) agent using Stable Baselines3 and Gymnasium. The chosen Atari environment is **ALE/Boxing-v5**.

The goal of this notebook is to:
- set up the required libraries
- confirm that the Boxing environment works correctly
- create and train a DQN agent
- save and load the trained model
- run evaluation
- compare multiple hyperparameter experiments

The notebook reflects the same workflow I used in my scripts and local setup.

## Installing Libraries



In [ ]:
!pip install "stable-baselines3[extra]" "gymnasium[atari,accept-rom-license]" ale-py shimmy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 8.8 MB/s eta 0:00:00


## Checking the Environment


In [ ]:
import gymnasium as gym
import ale_py

gym.register_envs(ale_py)

env = gym.make("ALE/Boxing-v5")
obs, info = env.reset()

print("Environment loaded successfully!")
print("Observation shape:", obs.shape)
print("Number of actions:", env.action_space.n)

env.close()

Environment loaded successfully!
Observation shape: (210, 160, 3)
Number of actions: 18


import gymnasium as gym
import ale_py

In [ ]:
import gymnasium as gym
import ale_py
from stable_baselines3 import DQN

gym.register_envs(ale_py)

env = gym.make("ALE/Boxing-v5")

model = DQN(
    "CnnPolicy",
    env,
    learning_rate=0.0005,
    buffer_size=10000,
    learning_starts=1000,
    batch_size=64,
    gamma=0.99,
    exploration_fraction=0.1,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.05,
    verbose=1
)

print("Model initialized successfully")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Model initialized successfully


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Training the Agent

In [ ]:
model.learn(total_timesteps=10000)

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 1.79e+03 |
|    ep_rew_mean      | -8       |
|    exploration_rate | 0.05     |
| time/               |          |
|    episodes         | 4        |
|    fps              | 148      |
|    time_elapsed     | 48       |
|    total_timesteps  | 7144     |
| train/              |          |
|    learning_rate    | 0.0005   |
|    loss             | 0.00155  |
|    n_updates        | 1535     |
----------------------------------


## Saving the Model



In [ ]:
model.save("solomon_dqn_boxing_model")

## Loading the Model


In [ ]:
from stable_baselines3 import DQN

loaded_model = DQN.load("solomon_dqn_boxing_model")
print("Model loaded successfully")

Model loaded successfully


## Evaluating the Agent


In [ ]:
import gymnasium as gym
import ale_py

gym.register_envs(ale_py)

eval_env = gym.make("ALE/Boxing-v5", render_mode="rgb_array")

obs, info = eval_env.reset()

for step in range(200):
    action, _ = loaded_model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)

    if terminated or truncated:
        obs, info = eval_env.reset()

eval_env.close()

## Hyperparameter Experiments


The main things I looked at were:
- whether the agent moved consistently
- whether it attacked more effectively
- whether it behaved randomly or more purposefully
- whether it improved or became too repetitive

In [ ]:
import pandas as pd

results = [
    ["Exp1", 0.0005, 0.95, 64, 1.0, 0.05, 0.1, -11, "random movement, weak performance"],
    ["Exp2", 0.0005, 0.95, 64, 1.0, 0.05, 0.1, -5.25, "better movement and more attacking"],
    ["Exp3", 0.0005, 0.95, 64, 1.0, 0.05, 0.1, 0, "good balance and improved behavior"],
    ["Exp4", 0.0001, 0.95, 64, 1.0, 0.05, 0.1, -6, "more controlled but missing punches"],
    ["Exp5", 0.0002, 0.95, 64, 1.0, 0.05, 0.1, -8, "slightly improved stability"],
    ["Exp6", 0.0005, 0.99, 64, 1.0, 0.05, 0.1, -4, "better long-term strategy"],
    ["Exp7", 0.0005, 0.90, 64, 1.0, 0.05, 0.1, -9, "less focus on future rewards"],
    ["Exp8", 0.0005, 0.95, 32, 1.0, 0.05, 0.1, -7, "less stable learning"],
    ["Exp9", 0.0005, 0.95, 128, 1.0, 0.05, 0.1, -6.5, "smoother but slower learning"],
    ["Exp10", 0.0005, 0.95, 64, 1.0, 0.01, 0.1, -3.5, "more precise but less exploration"]
]

columns = [
    "Experiment", "Learning Rate", "Gamma", "Batch Size",
    "Epsilon Start", "Epsilon End", "Epsilon Decay",
    "Final Reward", "Observed Behavior"
]

df = pd.DataFrame(results, columns=columns)
df

,Experiment,Learning Rate,Gamma,Batch Size,Epsilon Start,Epsilon End,Epsilon Decay,Final Reward,Observed Behavior
0,Exp1,0.0005,0.95,64,1.0,0.05,0.1,-11.00,"random movement, weak performance"
1,Exp2,0.0005,0.95,64,1.0,0.05,0.1,-5.25,better movement and more attacking
2,Exp3,0.0005,0.95,64,1.0,0.05,0.1,0.00,good balance and improved behavior
3,Exp4,0.0001,0.95,64,1.0,0.05,0.1,-6.00,more controlled but missing punches
4,Exp5,0.0002,0.95,64,1.0,0.05,0.1,-8.00,slightly improved stability
5,Exp6,0.0005,0.99,64,1.0,0.05,0.1,-4.00,better long-term strategy
6,Exp7,0.0005,0.90,64,1.0,0.05,0.1,-9.00,less focus on future rewards
7,Exp8,0.0005,0.95,32,1.0,0.05,0.1,-7.00,less stable learning
8,Exp9,0.0005,0.95,128,1.0,0.05,0.1,-6.50,smoother but slower learning
9,Exp10,0.0005,0.95,64,1.0,0.01,0.1,-3.50,more precise but less exploration


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Conclusion

From the experiments, I observed that different hyperparameter settings had a clear impact on the agent’s behavior. Increasing the gamma value improved long-term decision making, while adjusting the learning rate affected how quickly the agent learned.

A learning rate of 0.0005 combined with a gamma of 0.99 and batch size of 64 produced the best overall performance. The agent moved more effectively and attacked more consistently compared to other configurations.

I also observed that exploration settings played a key role. When epsilon was too low, the agent became repetitive and predictable. When it was too high, the agent behaved more randomly. A balanced value allowed the agent to improve while still exploring.

Overall, this experiment showed that tuning hyperparameters is essential for improving reinforcement learning performance. Even small changes can significantly affect how the agent behaves in the environment.